In [ ]:
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# CELL 1 — DOWNLOAD CROWDHUMAN FROM HUGGING FACE
# ============================================================

# Install the Hugging Face CLI
!pip install -q -U huggingface_hub

from huggingface_hub import snapshot_download
from pathlib import Path

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

REPO_ID = "sshao0516/CrowdHuman"
LOCAL_DIR = "/content/CrowdHuman"

print("=" * 60)
print("CrowdHuman Dataset Downloader")
print("=" * 60)

print(f"\nRepository : {REPO_ID}")
print(f"Download to: {LOCAL_DIR}")

# ------------------------------------------------------------
# Download the complete original dataset
# ------------------------------------------------------------

print("\nStarting download...")
print("⚠️ The complete dataset is approximately 14.2 GB.")
print("This may take some time depending on Colab's connection.\n")

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=LOCAL_DIR
)

print("\n" + "=" * 60)
print("DOWNLOAD COMPLETE ✅")
print("=" * 60)

# ------------------------------------------------------------
# Show downloaded files
# ------------------------------------------------------------

print("\nFiles downloaded:")

for file in sorted(Path(LOCAL_DIR).iterdir()):
    if file.is_file():
        size_gb = file.stat().st_size / (1024 ** 3)
        print(f"  {file.name:<30} {size_gb:.2f} GB")

In [ ]:
# ============================================================
# CELL 2 — VERIFY CROWDHUMAN ANNOTATIONS
# ============================================================

import json
from pathlib import Path

DATASET = Path("/content/CrowdHuman")

TRAIN_ANN = DATASET / "annotation_train.odgt"
VAL_ANN   = DATASET / "annotation_val.odgt"

print("=" * 70)
print("CrowdHuman Dataset Verification")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check required files
# ------------------------------------------------------------

required_files = [
    "CrowdHuman_train01.zip",
    "CrowdHuman_train02.zip",
    "CrowdHuman_train03.zip",
    "CrowdHuman_val.zip",
    "annotation_train.odgt",
    "annotation_val.odgt",
]

print("\n[1] Checking required files...\n")

all_present = True

for filename in required_files:
    path = DATASET / filename

    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"✅ {filename:<30} {size_mb:>10.2f} MB")
    else:
        print(f"❌ {filename:<30} MISSING")
        all_present = False

# ------------------------------------------------------------
# 2. Read one training annotation
# ------------------------------------------------------------

print("\n[2] Inspecting training annotation format...\n")

with open(TRAIN_ANN, "r") as f:
    first_line = f.readline()

annotation = json.loads(first_line)

print("Image ID:")
print(" ", annotation.get("ID"))

print("\nImage size:")
print(" ", annotation.get("width"), "x", annotation.get("height"))

print("\nNumber of ground-truth boxes:")
print(" ", len(annotation.get("gtboxes", [])))

# ------------------------------------------------------------
# 3. Inspect first few ground-truth boxes
# ------------------------------------------------------------

print("\n[3] Inspecting first 5 ground-truth annotations...\n")

for i, gtbox in enumerate(annotation.get("gtboxes", [])[:5]):

    print(f"Person/Box #{i + 1}")

    print("  tag :", gtbox.get("tag"))
    print("  fbox:", gtbox.get("fbox"))
    print()

# ------------------------------------------------------------
# 4. Count annotations
# ------------------------------------------------------------

print("[4] Counting training and validation images...\n")

def count_lines(annotation_file):

    count = 0

    with open(annotation_file, "r") as f:
        for line in f:
            if line.strip():
                count += 1

    return count


train_count = count_lines(TRAIN_ANN)
val_count = count_lines(VAL_ANN)

print(f"Training annotation entries  : {train_count}")
print(f"Validation annotation entries: {val_count}")

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)

if all_present:
    print("DATASET FILE CHECK: PASSED ✅")
else:
    print("DATASET FILE CHECK: FAILED ❌")

print("=" * 70)

In [ ]:
# ============================================================
# CELL 3 — EXTRACT CROWDHUMAN TRAIN + VALIDATION IMAGES
# ============================================================

import zipfile
from pathlib import Path
import shutil

DATASET = Path("/content/CrowdHuman")

# ------------------------------------------------------------
# Create extraction directories
# ------------------------------------------------------------

TRAIN_DIR = DATASET / "train_images"
VAL_DIR   = DATASET / "val_images"

TRAIN_DIR.mkdir(exist_ok=True)
VAL_DIR.mkdir(exist_ok=True)

print("=" * 70)
print("CrowdHuman Image Extraction")
print("=" * 70)

# ------------------------------------------------------------
# Training ZIP files
# ------------------------------------------------------------

train_zips = [
    DATASET / "CrowdHuman_train01.zip",
    DATASET / "CrowdHuman_train02.zip",
    DATASET / "CrowdHuman_train03.zip",
]

print("\n[1] Extracting TRAINING images")
print("-" * 70)

for zip_path in train_zips:

    print(f"\n📦 {zip_path.name}")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:

        members = zip_ref.namelist()

        print(f"   Files inside ZIP: {len(members)}")
        print("   Extracting...")

        zip_ref.extractall(TRAIN_DIR)

    print("   ✅ Done")


# ------------------------------------------------------------
# Validation ZIP
# ------------------------------------------------------------

print("\n[2] Extracting VALIDATION images")
print("-" * 70)

print(f"\n📦 {DATASET / 'CrowdHuman_val.zip'}")

with zipfile.ZipFile(DATASET / "CrowdHuman_val.zip", "r") as zip_ref:

    members = zip_ref.namelist()

    print(f"   Files inside ZIP: {len(members)}")
    print("   Extracting...")

    zip_ref.extractall(VAL_DIR)

print("   ✅ Done")


# ------------------------------------------------------------
# Count extracted image files
# ------------------------------------------------------------

def count_images(directory):

    extensions = {".jpg", ".jpeg", ".png"}

    return [
        path for path in directory.rglob("*")
        if path.suffix.lower() in extensions
    ]


train_images = count_images(TRAIN_DIR)
val_images = count_images(VAL_DIR)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXTRACTION SUMMARY")
print("=" * 70)

print(f"\nTraining images extracted   : {len(train_images)}")
print(f"Validation images extracted: {len(val_images)}")

print("\nExpected:")
print("Training images             : 15000")
print("Validation images           : 4370")

print("\n" + "=" * 70)

if len(train_images) == 15000:
    print("✅ Training extraction looks correct")
else:
    print("⚠️ Training image count differs from expected")

if len(val_images) == 4370:
    print("✅ Validation extraction looks correct")
else:
    print("⚠️ Validation image count differs from expected")

print("=" * 70)

In [ ]:
# ============================================================
# CELL 4 — VERIFY IMAGE / ANNOTATION MATCHING
# ============================================================
#
# Goal:
#   Make sure every CrowdHuman annotation has a corresponding
#   image before we start converting anything to YOLO format.
#
# CrowdHuman annotation:
#   ID = "284193,faa9000f2678b5e"
#
# Image:
#   284193,faa9000f2678b5e.jpg
#
# If these match correctly, we are ready for YOLO conversion.
# ============================================================

from pathlib import Path
import json

DATASET = Path("/content/CrowdHuman")

TRAIN_IMAGE_DIR = DATASET / "train_images"
VAL_IMAGE_DIR   = DATASET / "val_images"

TRAIN_ANN = DATASET / "annotation_train.odgt"
VAL_ANN   = DATASET / "annotation_val.odgt"


# ------------------------------------------------------------
# Build an index of all extracted images
# ------------------------------------------------------------

def build_image_index(image_directory):

    image_index = {}

    for image_path in image_directory.rglob("*"):

        if image_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:

            # Filename without extension
            image_id = image_path.stem

            image_index[image_id] = image_path

    return image_index


print("=" * 70)
print("IMAGE ↔ ANNOTATION MATCHING CHECK")
print("=" * 70)

print("\nBuilding image indexes...")

train_index = build_image_index(TRAIN_IMAGE_DIR)
val_index   = build_image_index(VAL_IMAGE_DIR)

print(f"Train images indexed: {len(train_index)}")
print(f"Val images indexed  : {len(val_index)}")


# ------------------------------------------------------------
# Check annotation IDs against image IDs
# ------------------------------------------------------------

def check_matching(annotation_file, image_index, split):

    total = 0
    matched = 0
    missing = []

    with open(annotation_file, "r") as f:

        for line in f:

            if not line.strip():
                continue

            annotation = json.loads(line)

            image_id = annotation["ID"]

            total += 1

            if image_id in image_index:
                matched += 1
            else:
                missing.append(image_id)

    print(f"\n{split.upper()} SET")
    print("-" * 50)
    print(f"Annotations : {total}")
    print(f"Matched     : {matched}")
    print(f"Missing     : {len(missing)}")

    if missing:
        print("\nFirst 10 missing image IDs:")

        for image_id in missing[:10]:
            print(" ", image_id)

    return total, matched, missing


train_total, train_matched, train_missing = check_matching(
    TRAIN_ANN,
    train_index,
    "train"
)

val_total, val_matched, val_missing = check_matching(
    VAL_ANN,
    val_index,
    "validation"
)


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL MATCHING RESULT")
print("=" * 70)

if (
    train_total == train_matched
    and val_total == val_matched
):

    print("\n✅ PERFECT MATCH!")
    print("Every annotation has a corresponding image.")
    print("\nWe are ready for:")
    print("CrowdHuman → YOLO format conversion 🚀")

else:

    print("\n⚠️ MATCHING PROBLEM DETECTED")

    print(f"Train missing: {len(train_missing)}")
    print(f"Val missing  : {len(val_missing)}")

print("=" * 70)

In [ ]:
# ============================================================
# CELL 5 — CROWDHUMAN → YOLO FORMAT CONVERSION
# ============================================================
#
# CrowdHuman provides bounding boxes in this format:
#
#     [x, y, width, height]
#
# YOLO requires:
#
#     class_id center_x center_y width height
#
# All coordinates except class_id must be normalized to 0-1.
#
# We use:
#
#     class_id = 0
#
# because our only class is:
#
#     0 = person
#
# We use CrowdHuman's "fbox" because it represents the
# full-body bounding box of a person.
# ============================================================

import json
import cv2
import shutil
from pathlib import Path
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------

DATASET = Path("/content/CrowdHuman")

OUTPUT = Path("/content/crowdhuman_yolo")

TRAIN_IMAGES = DATASET / "train_images"
VAL_IMAGES   = DATASET / "val_images"

TRAIN_ANN = DATASET / "annotation_train.odgt"
VAL_ANN   = DATASET / "annotation_val.odgt"


print("=" * 70)
print("CROWDHUMAN → YOLO DATASET CONVERSION")
print("=" * 70)

print("\nSource dataset:")
print(f"  {DATASET}")

print("\nYOLO dataset:")
print(f"  {OUTPUT}")


# ------------------------------------------------------------
# 2. CREATE YOLO DIRECTORY STRUCTURE
# ------------------------------------------------------------

print("\n[1] Creating YOLO directory structure...")

for split in ["train", "val"]:

    (OUTPUT / "images" / split).mkdir(
        parents=True,
        exist_ok=True
    )

    (OUTPUT / "labels" / split).mkdir(
        parents=True,
        exist_ok=True
    )

print("✅ Directory structure created")


# ------------------------------------------------------------
# 3. BUILD IMAGE INDEX
# ------------------------------------------------------------
#
# Instead of searching the filesystem for every annotation,
# we create a dictionary:
#
#     image_id → image_path
#
# Example:
#
#     "284193,faa9000f2678b5e"
#          ↓
#     ".../Images/284193,faa9000f2678b5e.jpg"
#
# ------------------------------------------------------------

def build_image_index(directory):

    print(f"\nIndexing images in:")
    print(f"  {directory}")

    image_index = {}

    for image_path in directory.rglob("*"):

        if image_path.suffix.lower() in {
            ".jpg",
            ".jpeg",
            ".png"
        }:

            image_id = image_path.stem

            image_index[image_id] = image_path

    print(f"✅ Found {len(image_index)} images")

    return image_index


train_index = build_image_index(TRAIN_IMAGES)
val_index   = build_image_index(VAL_IMAGES)


# ------------------------------------------------------------
# 4. CONVERT ANNOTATIONS
# ------------------------------------------------------------

def convert_split(annotation_file, image_index, split):

    print("\n" + "=" * 70)
    print(f"PROCESSING {split.upper()} SET")
    print("=" * 70)

    processed_images = 0
    missing_images = 0
    invalid_boxes = 0
    person_boxes = 0

    # --------------------------------------------------------
    # Read annotation file
    # --------------------------------------------------------

    with open(annotation_file, "r") as f:

        lines = f.readlines()

    print(f"\nAnnotations found: {len(lines)}")

    # --------------------------------------------------------
    # Process every image
    # --------------------------------------------------------

    for line in tqdm(
        lines,
        desc=f"Converting {split}"
    ):

        if not line.strip():
            continue

        # ----------------------------------------------------
        # Parse JSON annotation
        # ----------------------------------------------------

        annotation = json.loads(line)

        image_id = annotation["ID"]

        # ----------------------------------------------------
        # Find corresponding image
        # ----------------------------------------------------

        image_path = image_index.get(image_id)

        if image_path is None:

            missing_images += 1

            continue

        # ----------------------------------------------------
        # Read image
        # ----------------------------------------------------

        image = cv2.imread(str(image_path))

        if image is None:

            print(
                f"\n⚠️ Could not read image: "
                f"{image_path}"
            )

            continue

        image_height, image_width = image.shape[:2]

        # ----------------------------------------------------
        # Output paths
        # ----------------------------------------------------

        output_image = (
            OUTPUT
            / "images"
            / split
            / f"{image_id}.jpg"
        )

        output_label = (
            OUTPUT
            / "labels"
            / split
            / f"{image_id}.txt"
        )

        # ----------------------------------------------------
        # Copy image into YOLO dataset
        # ----------------------------------------------------

        #
        # We use shutil.copy2 instead of rewriting the image.
        # This preserves the original image quality.
        #

        shutil.copy2(
            image_path,
            output_image
        )

        # ----------------------------------------------------
        # Convert every person bounding box
        # ----------------------------------------------------

        yolo_labels = []

        for gtbox in annotation.get("gtboxes", []):

            # ------------------------------------------------
            # Keep only actual persons
            # ------------------------------------------------

            if gtbox.get("tag") != "person":
                continue

            # ------------------------------------------------
            # Get full-body bounding box
            # ------------------------------------------------

            fbox = gtbox.get("fbox")

            if fbox is None or len(fbox) != 4:

                invalid_boxes += 1

                continue

            x, y, box_width, box_height = fbox

            # ------------------------------------------------
            # Validate bounding box
            # ------------------------------------------------

            if box_width <= 0 or box_height <= 0:

                invalid_boxes += 1

                continue

            # ------------------------------------------------
            # CrowdHuman XYWH
            #
            #     x = left
            #     y = top
            #     w = width
            #     h = height
            #
            # YOLO needs center coordinates.
            # ------------------------------------------------

            center_x = x + (box_width / 2)
            center_y = y + (box_height / 2)

            # ------------------------------------------------
            # Normalize to 0-1
            # ------------------------------------------------

            center_x = center_x / image_width
            center_y = center_y / image_height

            box_width = box_width / image_width
            box_height = box_height / image_height

            # ------------------------------------------------
            # Sanity check
            # ------------------------------------------------

            if not (
                0 <= center_x <= 1
                and 0 <= center_y <= 1
                and 0 < box_width <= 1
                and 0 < box_height <= 1
            ):

                invalid_boxes += 1

                continue

            # ------------------------------------------------
            # YOLO label
            #
            # 0 = person
            #
            # Format:
            #
            # 0 center_x center_y width height
            # ------------------------------------------------

            yolo_labels.append(
                f"0 "
                f"{center_x:.6f} "
                f"{center_y:.6f} "
                f"{box_width:.6f} "
                f"{box_height:.6f}"
            )

            person_boxes += 1

        # ----------------------------------------------------
        # Save YOLO label file
        # ----------------------------------------------------

        with open(output_label, "w") as f:

            f.write(
                "\n".join(yolo_labels)
            )

        processed_images += 1

    # --------------------------------------------------------
    # Print split summary
    # --------------------------------------------------------

    print("\n" + "-" * 70)

    print(f"{split.upper()} SUMMARY")

    print("-" * 70)

    print(
        f"Images processed : {processed_images}"
    )

    print(
        f"Missing images   : {missing_images}"
    )

    print(
        f"Person boxes     : {person_boxes}"
    )

    print(
        f"Invalid boxes    : {invalid_boxes}"
    )


# ------------------------------------------------------------
# 5. CONVERT TRAINING SET
# ------------------------------------------------------------

convert_split(
    TRAIN_ANN,
    train_index,
    "train"
)


# ------------------------------------------------------------
# 6. CONVERT VALIDATION SET
# ------------------------------------------------------------

convert_split(
    VAL_ANN,
    val_index,
    "val"
)


# ------------------------------------------------------------
# 7. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONVERSION COMPLETE ✅")
print("=" * 70)

print("\nYOLO dataset created at:")

print(f"  {OUTPUT}")

print("\nStructure:")

print("""
crowdhuman_yolo/
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
""")

print("=" * 70)

In [ ]:
# ============================================================
# CELL 6 — VISUAL VERIFICATION OF YOLO LABELS
# ============================================================
#
# This is our final sanity check before training.
#
# We will:
#   1. Pick a random training image
#   2. Load its YOLO label file
#   3. Convert YOLO coordinates back to pixels
#   4. Draw the person bounding boxes
#   5. Display the image
#
# We expect to see GREEN/colored rectangles around people.
# ============================================================

import random
import cv2
import matplotlib.pyplot as plt
from pathlib import Path


# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------

YOLO_DATASET = Path("/content/crowdhuman_yolo")

IMAGE_DIR = YOLO_DATASET / "images" / "train"
LABEL_DIR = YOLO_DATASET / "labels" / "train"


# ------------------------------------------------------------
# Find images
# ------------------------------------------------------------

image_files = list(IMAGE_DIR.glob("*.jpg"))

print("=" * 70)
print("YOLO LABEL VISUAL VERIFICATION")
print("=" * 70)

print(f"\nTraining images available: {len(image_files)}")


# ------------------------------------------------------------
# Select random image
# ------------------------------------------------------------

random_image = random.choice(image_files)

image_id = random_image.stem

label_file = LABEL_DIR / f"{image_id}.txt"

print("\nSelected image:")
print(f"  {random_image.name}")

print("\nCorresponding label:")
print(f"  {label_file.name}")


# ------------------------------------------------------------
# Read image
# ------------------------------------------------------------

image = cv2.imread(str(random_image))

if image is None:
    raise RuntimeError("❌ Could not read image!")

image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

image_height, image_width = image.shape[:2]

print("\nImage dimensions:")
print(f"  Width  : {image_width}")
print(f"  Height : {image_height}")


# ------------------------------------------------------------
# Read YOLO labels
# ------------------------------------------------------------

if not label_file.exists():
    raise RuntimeError("❌ Label file does not exist!")

with open(label_file, "r") as f:
    lines = [line.strip() for line in f if line.strip()]

print("\nNumber of YOLO annotations:")
print(f"  {len(lines)}")


# ------------------------------------------------------------
# Draw bounding boxes
# ------------------------------------------------------------

boxes_drawn = 0

for line in lines:

    values = line.split()

    if len(values) != 5:
        continue

    class_id, center_x, center_y, width, height = map(
        float,
        values
    )

    # --------------------------------------------------------
    # YOLO normalized coordinates → pixel coordinates
    # --------------------------------------------------------

    center_x *= image_width
    center_y *= image_height

    width *= image_width
    height *= image_height

    x1 = int(center_x - width / 2)
    y1 = int(center_y - height / 2)

    x2 = int(center_x + width / 2)
    y2 = int(center_y + height / 2)

    # Keep coordinates inside image
    x1 = max(0, min(image_width - 1, x1))
    y1 = max(0, min(image_height - 1, y1))
    x2 = max(0, min(image_width - 1, x2))
    y2 = max(0, min(image_height - 1, y2))

    # --------------------------------------------------------
    # Draw bounding box
    # --------------------------------------------------------

    cv2.rectangle(
        image,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        2
    )

    # Label
    cv2.putText(
        image,
        "person",
        (x1, max(15, y1 - 5)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 255, 0),
        1,
        cv2.LINE_AA
    )

    boxes_drawn += 1


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print(f"\nBounding boxes drawn: {boxes_drawn}")

print("\nDisplaying image...")

plt.figure(figsize=(14, 9))
plt.imshow(image)
plt.axis("off")
plt.title(
    f"CrowdHuman — {image_id}\n"
    f"Person boxes: {boxes_drawn}"
)
plt.show()

In [ ]:
# ============================================================
# CELL 7 — CREATE YOLO DATASET CONFIGURATION
# ============================================================
#
# This file tells Ultralytics YOLO how our CrowdHuman dataset
# is organized.
#
# Dataset:
#
# /content/crowdhuman_yolo/
# ├── images/
# │   ├── train/
# │   └── val/
# │
# └── labels/
#     ├── train/
#     └── val/
#
# We have ONLY ONE detection class:
#
#     0 = person
#
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Dataset location
# ------------------------------------------------------------

YOLO_DATASET = Path("/content/crowdhuman_yolo")

DATA_YAML = YOLO_DATASET / "data.yaml"


print("=" * 70)
print("CREATING YOLO DATASET CONFIGURATION")
print("=" * 70)


# ------------------------------------------------------------
# Check required directories
# ------------------------------------------------------------

required_directories = [
    YOLO_DATASET / "images" / "train",
    YOLO_DATASET / "images" / "val",
    YOLO_DATASET / "labels" / "train",
    YOLO_DATASET / "labels" / "val",
]

print("\n[1] Checking dataset directories...\n")

for directory in required_directories:

    if directory.exists():

        print(f"✅ {directory}")

    else:

        print(f"❌ MISSING: {directory}")

        raise FileNotFoundError(
            f"Required directory does not exist: {directory}"
        )


# ------------------------------------------------------------
# Count images and labels
# ------------------------------------------------------------

print("\n[2] Counting images and labels...\n")


def count_files(directory, extensions):

    return sum(
        1
        for file in directory.iterdir()
        if file.is_file()
        and file.suffix.lower() in extensions
    )


train_images = count_files(
    YOLO_DATASET / "images" / "train",
    {".jpg", ".jpeg", ".png"}
)

val_images = count_files(
    YOLO_DATASET / "images" / "val",
    {".jpg", ".jpeg", ".png"}
)

train_labels = count_files(
    YOLO_DATASET / "labels" / "train",
    {".txt"}
)

val_labels = count_files(
    YOLO_DATASET / "labels" / "val",
    {".txt"}
)


print(f"Training images : {train_images}")
print(f"Training labels : {train_labels}")

print(f"\nValidation images : {val_images}")
print(f"Validation labels : {val_labels}")


# ------------------------------------------------------------
# Create data.yaml
# ------------------------------------------------------------

print("\n[3] Creating data.yaml...\n")


yaml_content = """path: /content/crowdhuman_yolo

train: images/train
val: images/val

names:
  0: person
"""


with open(DATA_YAML, "w") as file:
    file.write(yaml_content)


print("✅ data.yaml created")


# ------------------------------------------------------------
# Display YAML
# ------------------------------------------------------------

print("\n[4] Configuration:\n")
print("-" * 50)

print(DATA_YAML.read_text())

print("-" * 50)


# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

print("\n[5] Final dataset check...\n")

checks_passed = True


if train_images != 15000:
    print(
        f"⚠️ Expected 15000 training images, "
        f"found {train_images}"
    )
    checks_passed = False
else:
    print("✅ Training image count = 15000")


if val_images != 4370:
    print(
        f"⚠️ Expected 4370 validation images, "
        f"found {val_images}"
    )
    checks_passed = False
else:
    print("✅ Validation image count = 4370")


if train_labels != 15000:
    print(
        f"⚠️ Expected 15000 training labels, "
        f"found {train_labels}"
    )
    checks_passed = False
else:
    print("✅ Training label count = 15000")


if val_labels != 4370:
    print(
        f"⚠️ Expected 4370 validation labels, "
        f"found {val_labels}"
    )
    checks_passed = False
else:
    print("✅ Validation label count = 4370")


print("\n" + "=" * 70)

if checks_passed:

    print("🎉 DATASET CONFIGURATION PASSED!")

    print("\nReady for YOLO validation/training.")

else:

    print("⚠️ SOME CHECKS FAILED.")

print("=" * 70)

In [ ]:
# ============================================================
# CELL 8 — ULTRALYTICS DATASET VERIFICATION
# ============================================================
#
# This is the FINAL CHECK before training.
#
# We already verified:
#   ✅ 15,000 training images
#   ✅ 4,370 validation images
#   ✅ 15,000 training labels
#   ✅ 4,370 validation labels
#   ✅ Bounding boxes visually look correct
#
# Now we let Ultralytics verify the dataset itself.
# ============================================================

from ultralytics import YOLO
from pathlib import Path

DATA_YAML = "/content/crowdhuman_yolo/data.yaml"

print("=" * 70)
print("ULTRALYTICS DATASET VERIFICATION")
print("=" * 70)

print("\nDataset configuration:")
print(f"  {DATA_YAML}")

# ------------------------------------------------------------
# Load YOLO26m
# ------------------------------------------------------------

print("\n[1] Loading YOLO26m pretrained model...")

model = YOLO("yolo26m.pt")

print("✅ YOLO26m loaded successfully")


# ------------------------------------------------------------
# Run validation
# ------------------------------------------------------------
#
# IMPORTANT:
# We are NOT training here.
#
# We are only asking Ultralytics to load the validation
# dataset and process it.
#
# This allows us to catch dataset-format problems before
# starting the expensive training process.
# ------------------------------------------------------------

print("\n[2] Running dataset validation...")
print("This may take a little while.\n")

try:

    validation_results = model.val(
        data=DATA_YAML,
        split="val",
        imgsz=640,
        batch=8,
        workers=2,
        device=0,
        plots=False,
        verbose=True
    )

    print("\n" + "=" * 70)
    print("ULTRALYTICS DATASET VALIDATION COMPLETED ✅")
    print("=" * 70)

    print("\nUltralytics successfully loaded the validation dataset.")

except Exception as e:

    print("\n" + "=" * 70)
    print("❌ DATASET VALIDATION FAILED")
    print("=" * 70)

    print("\nError:")
    print(e)

    raise

In [ ]:
# ============================================================
# CELL 9 — YOLO26m FINE-TUNING ON CROWDHUMAN
# ============================================================
#
# Goal:
#   Fine-tune the pretrained YOLO26m model so that it becomes
#   better at detecting PERSONS in crowded scenes.
#
# Starting model:
#   YOLO26m pretrained on COCO
#
# Training dataset:
#   CrowdHuman
#
# Classes:
#   0 = person
#
# Hardware detected earlier:
#   NVIDIA Tesla T4 - ~15 GB VRAM
#
# ============================================================

from ultralytics import YOLO
import torch
from pathlib import Path


# ------------------------------------------------------------
# 1. CHECK GPU
# ------------------------------------------------------------

print("=" * 70)
print("YOLO26m — CROWDHUMAN FINE-TUNING")
print("=" * 70)

print("\n[1] Hardware check")

if torch.cuda.is_available():

    gpu_name = torch.cuda.get_device_name(0)

    gpu_memory = torch.cuda.get_device_properties(0).total_memory
    gpu_memory_gb = gpu_memory / (1024 ** 3)

    print(f"✅ GPU: {gpu_name}")
    print(f"✅ VRAM: {gpu_memory_gb:.2f} GB")

else:

    raise RuntimeError(
        "❌ CUDA GPU is not available. "
        "Enable a GPU runtime in Colab."
    )


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

DATA_YAML = "/content/crowdhuman_yolo/data.yaml"

PROJECT_DIR = "/content/runs/crowdhuman"

RUN_NAME = "yolo26m_v1"


print("\n[2] Training configuration")

print(f"Dataset : {DATA_YAML}")
print(f"Output  : {PROJECT_DIR}/{RUN_NAME}")


# ------------------------------------------------------------
# 3. LOAD PRETRAINED YOLO26m
# ------------------------------------------------------------

print("\n[3] Loading pretrained YOLO26m...")

model = YOLO("yolo26m.pt")

print("✅ Pretrained YOLO26m loaded")


# ------------------------------------------------------------
# 4. START FINE-TUNING
# ------------------------------------------------------------

print("\n[4] Starting fine-tuning...")
print()
print("Training:")
print("  Model       : YOLO26m")
print("  Dataset     : CrowdHuman")
print("  Classes     : person")
print("  Epochs      : 50")
print("  Image size  : 640")
print("  Batch size  : 8")
print("  Device      : Tesla T4")
print()
print("⚠️ This will take some time.")
print("⚠️ Do not interrupt the cell unless an error occurs.")
print()


results = model.train(

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    data=DATA_YAML,

    # --------------------------------------------------------
    # Training duration
    # --------------------------------------------------------

    epochs=50,

    # --------------------------------------------------------
    # Image resolution
    # --------------------------------------------------------

    imgsz=640,

    # --------------------------------------------------------
    # Tesla T4-friendly batch size
    # --------------------------------------------------------

    batch=8,

    # --------------------------------------------------------
    # DataLoader workers
    # --------------------------------------------------------

    workers=2,

    # --------------------------------------------------------
    # GPU
    # --------------------------------------------------------

    device=0,

    # --------------------------------------------------------
    # Pretrained weights
    # --------------------------------------------------------

    pretrained=True,

    # --------------------------------------------------------
    # Stop if validation doesn't improve
    # --------------------------------------------------------

    patience=10,

    # --------------------------------------------------------
    # Save training results
    # --------------------------------------------------------

    project=PROJECT_DIR,
    name=RUN_NAME,

    # --------------------------------------------------------
    # Save best + last checkpoints
    # --------------------------------------------------------

    save=True,

    # --------------------------------------------------------
    # Validate after every epoch
    # --------------------------------------------------------

    val=True,

    # --------------------------------------------------------
    # Save training plots
    # --------------------------------------------------------

    plots=True,

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    seed=42,

    # --------------------------------------------------------
    # Automatically use suitable optimizer
    # --------------------------------------------------------

    optimizer="auto",

    # --------------------------------------------------------
    # Maximum detections per image during validation
    #
    # CrowdHuman can contain >300 people in one image.
    # --------------------------------------------------------

    max_det=500,

    # --------------------------------------------------------
    # Close Mosaic augmentation during final epochs
    # --------------------------------------------------------

    close_mosaic=10,

    # --------------------------------------------------------
    # Use mixed precision on the T4
    # --------------------------------------------------------

    amp=True,

    # --------------------------------------------------------
    # Verbose training output
    # --------------------------------------------------------

    verbose=True
)


# ------------------------------------------------------------
# 5. TRAINING COMPLETE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🎉 YOLO26m FINE-TUNING COMPLETE")
print("=" * 70)


# ------------------------------------------------------------
# 6. SHOW MODEL LOCATIONS
# ------------------------------------------------------------

run_dir = Path(PROJECT_DIR) / RUN_NAME

best_model = run_dir / "weights" / "best.pt"
last_model = run_dir / "weights" / "last.pt"

print("\nBest model:")
print(f"  {best_model}")

print("\nLast model:")
print(f"  {last_model}")

print("\nTraining results:")
print(f"  {run_dir}")

print("\n" + "=" * 70)

## Restoring the model

In [ ]:
# ============================================================
# RESTORE YOLO26m V1 CHECKPOINT
# ============================================================

from pathlib import Path
import zipfile
import shutil

BACKUP_ZIP = "/content/yolo26m_v1_backup_22.zip"
RESTORE_DIR = "/content"

print("=" * 65)
print("RESTORING YOLO26m V1 TRAINING CHECKPOINT")
print("=" * 65)

# Check backup
if not Path(BACKUP_ZIP).exists():
    raise FileNotFoundError(f"Backup not found: {BACKUP_ZIP}")

print(f"\n✅ Backup found: {BACKUP_ZIP}")
print(f"📦 Backup size: {Path(BACKUP_ZIP).stat().st_size / (1024**2):.1f} MB")

# Extract
print("\n[1] Extracting backup...")
with zipfile.ZipFile(BACKUP_ZIP, "r") as z:
    z.extractall(RESTORE_DIR)

print("✅ Extraction complete")

# Locate restored run
RUN_DIR = Path("/content/runs/crowdhuman/yolo26m_v1")
WEIGHTS_DIR = RUN_DIR / "weights"

print("\n[2] Checking restored files...")

required = [
    WEIGHTS_DIR / "last.pt",
    WEIGHTS_DIR / "best.pt",
    RUN_DIR / "results.csv",
    RUN_DIR / "args.yaml",
]

for file in required:
    if file.exists():
        size = file.stat().st_size / (1024**2)
        print(f"✅ {file.relative_to('/content')} ({size:.1f} MB)")
    else:
        print(f"❌ MISSING: {file}")

print("\n" + "=" * 65)
print("RESTORE CHECK COMPLETE")
print("=" * 65)

print(f"\n📁 Run directory:")
print(f"   {RUN_DIR}")

print(f"\n⭐ Resume checkpoint:")
print(f"   {WEIGHTS_DIR / 'last.pt'}")

In [ ]:
# ============================================================
# RESTORE CROWDHUMAN DATASET
# ============================================================

!pip install -q -U huggingface_hub

from huggingface_hub import snapshot_download
from pathlib import Path

REPO_ID = "sshao0516/CrowdHuman"
DATA_DIR = "/content/CrowdHuman"

print("=" * 65)
print("RESTORING CROWDHUMAN DATASET")
print("=" * 65)

print("\nDownloading required training/validation files...")
print("⏳ This is approximately 12 GB, so it may take some time.\n")

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=DATA_DIR,
    allow_patterns=[
        "CrowdHuman_train01.zip",
        "CrowdHuman_train02.zip",
        "CrowdHuman_train03.zip",
        "CrowdHuman_val.zip",
        "annotation_train.odgt",
        "annotation_val.odgt",
        "README.md"
    ]
)

print("\n" + "=" * 65)
print("DATASET DOWNLOAD COMPLETE")
print("=" * 65)

# Verify required files
required_files = [
    "CrowdHuman_train01.zip",
    "CrowdHuman_train02.zip",
    "CrowdHuman_train03.zip",
    "CrowdHuman_val.zip",
    "annotation_train.odgt",
    "annotation_val.odgt"
]

print("\nChecking required files:")

for name in required_files:
    path = Path(DATA_DIR) / name
    if path.exists():
        size_gb = path.stat().st_size / (1024**3)
        print(f"✅ {name:<30} {size_gb:.2f} GB")
    else:
        print(f"❌ MISSING: {name}")

print("\n📁 Dataset directory:")
print(f"   {DATA_DIR}")

In [ ]:
# ============================================================
# REBUILD CROWDHUMAN YOLO DATASET
# ============================================================

from pathlib import Path
import zipfile
import shutil
import json

SOURCE_DIR = Path("/content/CrowdHuman")
YOLO_DIR = Path("/content/crowdhuman_yolo")

TRAIN_IMG_DIR = YOLO_DIR / "images" / "train"
VAL_IMG_DIR   = YOLO_DIR / "images" / "val"
TRAIN_LBL_DIR = YOLO_DIR / "labels" / "train"
VAL_LBL_DIR   = YOLO_DIR / "labels" / "val"

print("=" * 70)
print("REBUILDING CROWDHUMAN YOLO DATASET")
print("=" * 70)

# ------------------------------------------------------------
# 1. Create directories
# ------------------------------------------------------------

for directory in [
    TRAIN_IMG_DIR,
    VAL_IMG_DIR,
    TRAIN_LBL_DIR,
    VAL_LBL_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("\n[1] Directories ready ✅")


# ------------------------------------------------------------
# 2. Extract training and validation ZIPs
# ------------------------------------------------------------

train_zips = [
    SOURCE_DIR / "CrowdHuman_train01.zip",
    SOURCE_DIR / "CrowdHuman_train02.zip",
    SOURCE_DIR / "CrowdHuman_train03.zip",
]

val_zip = SOURCE_DIR / "CrowdHuman_val.zip"

TRAIN_EXTRACT = SOURCE_DIR / "train_extracted"
VAL_EXTRACT = SOURCE_DIR / "val_extracted"

TRAIN_EXTRACT.mkdir(exist_ok=True)
VAL_EXTRACT.mkdir(exist_ok=True)

print("\n[2] Extracting training ZIPs...")

for i, zip_path in enumerate(train_zips, 1):
    print(f"   Extracting train{i}: {zip_path.name}")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(TRAIN_EXTRACT)

print("   Training extraction complete ✅")

print("\n   Extracting validation ZIP...")

with zipfile.ZipFile(val_zip, "r") as z:
    z.extractall(VAL_EXTRACT)

print("   Validation extraction complete ✅")


# ------------------------------------------------------------
# 3. Find extracted images
# ------------------------------------------------------------

print("\n[3] Searching extracted images...")

train_images = list(TRAIN_EXTRACT.rglob("*.jpg"))
val_images = list(VAL_EXTRACT.rglob("*.jpg"))

print(f"   Training images found   : {len(train_images)}")
print(f"   Validation images found : {len(val_images)}")


# ------------------------------------------------------------
# 4. Build image indexes
# ------------------------------------------------------------

train_index = {
    p.stem: p
    for p in train_images
}

val_index = {
    p.stem: p
    for p in val_images
}

print("\n[4] Image indexes created ✅")


# ------------------------------------------------------------
# 5. Load CrowdHuman annotations
# ------------------------------------------------------------

TRAIN_ANN = SOURCE_DIR / "annotation_train.odgt"
VAL_ANN = SOURCE_DIR / "annotation_val.odgt"

print("\n[5] Loading annotations...")

def load_odgt(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


train_annotations = load_odgt(TRAIN_ANN)
val_annotations = load_odgt(VAL_ANN)

print(f"   Training annotations   : {len(train_annotations)}")
print(f"   Validation annotations : {len(val_annotations)}")


# ------------------------------------------------------------
# 6. Convert CrowdHuman boxes → YOLO format
# ------------------------------------------------------------

def convert_annotations(
    annotations,
    image_index,
    output_image_dir,
    output_label_dir,
    split_name
):
    processed = 0
    missing = 0
    person_boxes = 0
    invalid_boxes = 0
    empty_images = 0

    for ann in annotations:

        image_id = ann["ID"]

        # Find source image
        source_image = image_index.get(image_id)

        if source_image is None:
            missing += 1
            continue

        # Read image dimensions
        from PIL import Image

        with Image.open(source_image) as img:
            img_width, img_height = img.size

        yolo_lines = []

        for gtbox in ann.get("gtboxes", []):

            # CrowdHuman uses string tags:
            # "person" = valid person
            # "mask"   = ignore
            if gtbox.get("tag") != "person":
                continue

            # Full-body bounding box
            fbox = gtbox.get("fbox")

            if not fbox or len(fbox) != 4:
                invalid_boxes += 1
                continue

            x, y, w, h = map(float, fbox)

            # Sanity check
            if w <= 0 or h <= 0:
                invalid_boxes += 1
                continue

            # Clip box to image boundaries
            x1 = max(0.0, x)
            y1 = max(0.0, y)
            x2 = min(float(img_width), x + w)
            y2 = min(float(img_height), y + h)

            new_w = x2 - x1
            new_h = y2 - y1

            if new_w <= 0 or new_h <= 0:
                invalid_boxes += 1
                continue

            # YOLO format
            cx = (x1 + x2) / 2.0
            cy = (y1 + y2) / 2.0

            cx /= img_width
            cy /= img_height
            new_w /= img_width
            new_h /= img_height

            # Clamp normalized values
            cx = min(max(cx, 0.0), 1.0)
            cy = min(max(cy, 0.0), 1.0)
            new_w = min(max(new_w, 0.0), 1.0)
            new_h = min(max(new_h, 0.0), 1.0)

            yolo_lines.append(
                f"0 {cx:.6f} {cy:.6f} {new_w:.6f} {new_h:.6f}"
            )

            person_boxes += 1

        # Copy image
        destination_image = output_image_dir / source_image.name
        shutil.copy2(source_image, destination_image)

        # Write label
        destination_label = output_label_dir / f"{source_image.stem}.txt"

        with open(destination_label, "w", encoding="utf-8") as f:
            f.write("\n".join(yolo_lines))

        processed += 1

        if not yolo_lines:
            empty_images += 1

        # Progress
        if processed % 1000 == 0:
            print(
                f"   {split_name}: "
                f"{processed}/{len(annotations)} processed"
            )

    return {
        "processed": processed,
        "missing": missing,
        "person_boxes": person_boxes,
        "invalid_boxes": invalid_boxes,
        "empty_images": empty_images
    }


# ------------------------------------------------------------
# 7. Convert TRAIN
# ------------------------------------------------------------

print("\n[6] Converting TRAIN annotations → YOLO...")

train_stats = convert_annotations(
    train_annotations,
    train_index,
    TRAIN_IMG_DIR,
    TRAIN_LBL_DIR,
    "TRAIN"
)


# ------------------------------------------------------------
# 8. Convert VAL
# ------------------------------------------------------------

print("\n[7] Converting VAL annotations → YOLO...")

val_stats = convert_annotations(
    val_annotations,
    val_index,
    VAL_IMG_DIR,
    VAL_LBL_DIR,
    "VAL"
)


# ------------------------------------------------------------
# 9. Create data.yaml
# ------------------------------------------------------------

DATA_YAML = YOLO_DIR / "data.yaml"

DATA_YAML.write_text(
"""path: /content/crowdhuman_yolo

train: images/train
val: images/val

names:
  0: person
""",
encoding="utf-8"
)


# ------------------------------------------------------------
# 10. Final report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET REBUILD COMPLETE")
print("=" * 70)

print("\nTRAIN")
print(f"  Images processed : {train_stats['processed']}")
print(f"  Missing images   : {train_stats['missing']}")
print(f"  Person boxes     : {train_stats['person_boxes']}")
print(f"  Invalid boxes    : {train_stats['invalid_boxes']}")
print(f"  Empty labels     : {train_stats['empty_images']}")

print("\nVAL")
print(f"  Images processed : {val_stats['processed']}")
print(f"  Missing images   : {val_stats['missing']}")
print(f"  Person boxes     : {val_stats['person_boxes']}")
print(f"  Invalid boxes    : {val_stats['invalid_boxes']}")
print(f"  Empty labels     : {val_stats['empty_images']}")

print("\nYOLO dataset:")
print(f"  {YOLO_DIR}")

print("\ndata.yaml:")
print(f"  {DATA_YAML}")

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# VERIFY RESTORED DATASET + CHECKPOINT
# ============================================================

from pathlib import Path
import csv

YOLO_DIR = Path("/content/crowdhuman_yolo")
RUN_DIR = Path("/content/runs/crowdhuman/yolo26m_v1")

print("=" * 70)
print("FINAL PRE-RESUME VERIFICATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Dataset structure
# ------------------------------------------------------------

paths = {
    "Train images": YOLO_DIR / "images/train",
    "Val images": YOLO_DIR / "images/val",
    "Train labels": YOLO_DIR / "labels/train",
    "Val labels": YOLO_DIR / "labels/val",
    "data.yaml": YOLO_DIR / "data.yaml",
}

print("\n[1] Dataset structure")

for name, path in paths.items():
    if path.exists():
        print(f"✅ {name:<18}: {path}")
    else:
        print(f"❌ MISSING {name}: {path}")


# ------------------------------------------------------------
# 2. Count files
# ------------------------------------------------------------

train_images = list((YOLO_DIR / "images/train").glob("*.jpg"))
val_images = list((YOLO_DIR / "images/val").glob("*.jpg"))

train_labels = list((YOLO_DIR / "labels/train").glob("*.txt"))
val_labels = list((YOLO_DIR / "labels/val").glob("*.txt"))

print("\n[2] File counts")

print(f"Train images : {len(train_images)}")
print(f"Train labels : {len(train_labels)}")
print(f"Val images   : {len(val_images)}")
print(f"Val labels   : {len(val_labels)}")


# ------------------------------------------------------------
# 3. Check checkpoint
# ------------------------------------------------------------

best_pt = RUN_DIR / "weights/best.pt"
last_pt = RUN_DIR / "weights/last.pt"

print("\n[3] Checkpoints")

for name, path in [
    ("best.pt", best_pt),
    ("last.pt", last_pt),
]:
    if path.exists():
        size = path.stat().st_size / (1024**2)
        print(f"✅ {name:<8}: {size:.1f} MB")
    else:
        print(f"❌ MISSING: {path}")


# ------------------------------------------------------------
# 4. Read previous training history
# ------------------------------------------------------------

results_csv = RUN_DIR / "results.csv"

print("\n[4] Previous training history")

if results_csv.exists():
    with open(results_csv, newline="") as f:
        rows = list(csv.DictReader(f))

    print(f"Recorded epochs: {len(rows)}")

    if rows:
        last_row = rows[-1]

        print(f"Last recorded epoch: {last_row.get('epoch', 'unknown')}")

        for key in [
            "metrics/precision(B)",
            "metrics/recall(B)",
            "metrics/mAP50(B)",
            "metrics/mAP50-95(B)",
        ]:
            if key in last_row:
                print(f"{key:<28}: {last_row[key]}")
else:
    print("⚠️ results.csv not found")


# ------------------------------------------------------------
# 5. Final decision
# ------------------------------------------------------------

print("\n" + "=" * 70)

if (
    len(train_images) == 15000
    and len(train_labels) == 15000
    and len(val_images) == 4370
    and len(val_labels) == 4370
    and best_pt.exists()
    and last_pt.exists()
):
    print("✅ EVERYTHING IS READY")
    print("✅ Dataset restored")
    print("✅ Checkpoints restored")
    print("✅ Ready to resume training")
else:
    print("❌ SOMETHING IS MISSING — DO NOT RESUME YET")

print("=" * 70)

In [ ]:
# ============================================================
# RESUME YOLO26m V1 FROM EPOCH 22
# ============================================================

from ultralytics import YOLO
from pathlib import Path

LAST_CHECKPOINT = "/content/runs/crowdhuman/yolo26m_v1/weights/last.pt"

print("=" * 70)
print("RESUMING YOLO26m V1 TRAINING")
print("=" * 70)

print(f"\nCheckpoint:")
print(f"  {LAST_CHECKPOINT}")

if not Path(LAST_CHECKPOINT).exists():
    raise FileNotFoundError("❌ last.pt not found!")

print("\n✅ Checkpoint found")
print("⏳ Loading saved training state...")

model = YOLO(LAST_CHECKPOINT)

print("✅ Model loaded")
print("\n🚀 Resuming training...")
print("   Previous progress : Epoch 22/50")
print("   Target            : Epoch 50")
print("   Resume mode       : ON")
print("=" * 70)

results = model.train(
    resume=True
)

print("\n" + "=" * 70)
print("TRAINING SESSION FINISHED")
print("=" * 70)